In [1]:
import ray

@ray.remote
class Counter:
    def __init__(self):
        self.value = 0

    def increment(self):
        self.value += 1
        return self.value

    def get_counter(self):
        return self.value

# Create an actor from this class.
counter = Counter.remote()

2026-02-28 11:07:02,578	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/Users/yinyajun/miniconda3/envs/tomo/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [2]:
!ray list actors


======== List: 2026-02-28 11:07:05.242952 ========
Stats:
------------------------------
Total: 1

Table:
------------------------------
    ACTOR_ID                          CLASS_NAME    STATE      JOB_ID  NAME    NODE_ID                                                     PID  RAY_NAMESPACE
 0  7ca7f27ea0e98784d73e620901000000  Counter       ALIVE    01000000          a0d3e018a2ec08881839d0bb556f525d8399d2a075587b00e3e409fa  14428  4f6e69a7-faa3-4e07-9e9b-00c11085a137



In [3]:
# Call the actor.
obj_ref = counter.increment.remote()
print(ray.get(obj_ref))

1


In [4]:
# Create ten Counter actors.
counters = [Counter.remote() for _ in range(10)]

# Increment each Counter once and get the results. These tasks all happen in
# parallel.
results = ray.get([c.increment.remote() for c in counters])
print(results)

# Increment the first Counter five times. These tasks are executed serially
# and share state.
results = ray.get([counters[0].increment.remote() for _ in range(5)])
print(results)

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[2, 3, 4, 5, 6]


In [5]:
import time

@ray.remote
def f(counter):
    for _ in range(10):
        time.sleep(0.1)
        counter.increment.remote()

In [6]:
counter = Counter.remote()

# Start some tasks that use the actor.
[f.remote(counter) for _ in range(3)]

# Print the counter value.
for _ in range(10):
    time.sleep(0.1)
    print(ray.get(counter.get_counter.remote()))

0
0
4
9
12
15
18
21
24
27


## 它主要说了三件事

### 1) Actor 用 `ray.remote(MyClass)`，别用 `@ray.remote` 装饰类

- **不推荐**：

  ```
  @ray.remote
  class Counter: ...
  ```

- **推荐**：

  ```
  CounterActor = ray.remote(Counter)
  ```

原因：`ray.remote(MyClass)` 能更好地保留原始 `Counter` 类的类型信息，IDE/类型检查器更容易知道 actor 里有哪些方法、返回什么类型。

------

### 2) Actor 方法用 `@ray.method` 装饰，提升远程调用的类型提示

```
@ray.method
def increment(self) -> int: ...
```

这样当你写 `counter.increment.remote()` 的时候，类型检查器能推断出：这个 remote 调用会返回 `ObjectRef[int]`。

------

### 3) 给 actor 类和 actor handle 加上泛型类型注解

- `ActorClass[MyClass]`：表示“这是一个 Counter 的 actor 类（工厂）”
- `ActorProxy[MyClass]`：表示“这是一个 Counter actor 的句柄（handle）”

示例里：

```
CounterActor: ActorClass[Counter] = ray.remote(Counter)
counter: ActorProxy[Counter] = CounterActor.remote()
obj_ref: ray.ObjectRef[int] = counter.increment.remote()
```

好处：IDE 会给你 `counter.` 的方法补全；类型检查器知道 `increment` 最后会得到 `int`（通过 `ray.get(obj_ref)`）。


In [7]:
import ray
from ray.actor import ActorClass, ActorProxy

class Counter:
    def __init__(self):
        self.value = 0

    @ray.method
    def increment(self) -> int:
        self.value += 1
        return self.value

CounterActor: ActorClass[Counter] = ray.remote(Counter)
counter: ActorProxy[Counter] = CounterActor.remote()

# Type checkers and IDEs will now provide type hints for remote methods
obj_ref: ray.ObjectRef[int] = counter.increment.remote()
print(ray.get(obj_ref))

1


In [8]:
import ray
import asyncio
import time


@ray.remote
class Actor:
    async def f(self):
        try:
            await asyncio.sleep(5)
        except asyncio.CancelledError:
            print("Actor task canceled.")


actor = Actor.remote()
ref = actor.f.remote()

# Wait until task is scheduled.
time.sleep(1)
ray.cancel(ref)

try:
    ray.get(ref)
except ray.exceptions.RayTaskError:
    print("Object reference was cancelled.")

(Actor pid=14444) Actor task canceled.
Object reference was cancelled.


1) 还没被调度的任务（Unscheduled）

如果 task 还在队列里、Ray 还没把它派到某个 worker/actor 上执行：

Ray 会尝试直接取消“调度这件事”

如果成功，你后面 ray.get(actor_task_ref) 会抛 TaskCancelledError
意思：它甚至没开始跑，就被取消了。

2) 正在跑的 actor task（普通 actor / threaded actor）

如果 task 已经在 单线程 actor 或 多线程 actor 上开始执行了：

Ray 不一定能把它“立刻打断”，而是设置一个 取消标记

你需要在任务代码里 主动检查：

ray.get_runtime_context().is_canceled()

这样你可以做“优雅退出”：比如循环里定期检查，一旦被取消就清理资源并 return/raise。

3) 正在跑的 async actor task（异步 actor）

如果是 async actor（方法是 async def）：

Ray 会尝试取消对应的 asyncio.Task（按 asyncio 的取消语义）

但如果你的 async 函数里一直不 await，它不会在中途被打断（因为 asyncio 的取消通常在 await 点生效）

且 ray.get_runtime_context().is_canceled() 对 async actor 不支持，调用会直接 RuntimeError

4) 取消不保证一定成功（best-effort）

Ray 会尽力取消，但不承诺 100% 成功。例如取消请求没能送达执行器，任务可能还是跑了。

你可以通过 ray.get(actor_task_ref) 看最终结果：如果真的取消了会报 TaskCancelledError，否则可能正常返回/报别的错。

5) 递归取消（recursive=True）

Ray 会跟踪一个 task 派生出来的子任务/子 actor task。

你如果 recursive=True，它会把 所有子任务一起取消，而不只是当前这一个。

一句话总结

没开始的任务可以直接取消；开始跑的普通/threaded actor 需要任务内部配合检查取消标记；async actor 用 asyncio 的方式取消但必须有 await 才“打得断”；而且取消永远是尽力而为，不保证一定成功；recursive=True 会连带取消子任务。

你可以把它理解成三种 actor 执行模型之一：

## 1) Regular actor（普通/单线程 actor）

- 默认模式
- 同一时间只跑 **一个** actor 方法调用
- 其他调用会在 actor 的队列里排队

```
@ray.remote
class A:
    def f(self): ...
```

## 2) Threaded actor（线程化 actor）

- 仍然是 **同步方法**（`def`），但允许并发
- Ray 会在这个 actor 的 worker 进程里用 **线程池** 同时执行多个调用
- 典型开启方式：设置 `max_concurrency > 1`

```
@ray.remote(max_concurrency=8)
class A:
    def f(self): ...
```

这时 `a.f.remote()` 可以在同一个 actor 里并发跑最多 8 个（大致上就是线程并发）。

### 适用场景

- I/O 密集：网络请求、磁盘读写、等待外部服务
- 你希望一个 actor 同时处理多个请求（类似“一个服务实例，多线程处理”）

### 注意点（坑）

- 线程并发意味着：**actor 内部共享状态需要加锁**（`threading.Lock` 等），否则会有竞态条件
- CPU 密集任务：Python 还受 GIL 影响，多线程不一定加速（但 I/O 一般没问题）

## 3) Async actor（异步 actor）

- 方法是 `async def`
- 并发靠 asyncio event loop（协程并发），不是线程

```
@ray.remote
class A:
    async def f(self): ...
```

------



In [9]:
import ray
import time


@ray.remote
class SyncActor:
    def __init__(self):
        self.is_canceled = False

    def long_running_method(self):
        """A sync actor method that checks for cancellation periodically."""
        for i in range(100):
            # For sync actor tasks, is_canceled() can be checked in the task body
            if ray.get_runtime_context().is_canceled():
                self.is_canceled = True
                print("Actor task canceled, cleaning up...")
                return "canceled"
            time.sleep(0.1)
        return "completed"

    def get_cancel_status(self):
        return self.is_canceled


# Sync actor task cancellation with periodic checking
actor = SyncActor.remote()
actor_task_ref = actor.long_running_method.remote()

# Wait until task is scheduled.
time.sleep(1)
ray.cancel(actor_task_ref)

# The TaskCancelledError will be raised when calling ray.get
try:
    result = ray.get(actor_task_ref)
except ray.exceptions.TaskCancelledError:
    print("Actor task was cancelled")

# The get_cancel_status will return True after cancellation
cancel_status = ray.get(actor.get_cancel_status.remote())
print(f"Actor detected cancellation: {cancel_status}")

(SyncActor pid=14451) Actor task canceled, cleaning up...
Actor task was cancelled
Actor detected cancellation: True


对于 **非 async 的 actor task**（普通 `def` 方法）：

- `ray.cancel(ref)` **不会把正在执行的函数“强行打断”**（不能像杀线程一样中断）

- Ray 只是发出一个“取消请求”

- 你要在任务内部 **定期检查**：

  ```
  ray.get_runtime_context().is_canceled()
  ```

- 检测到取消后，你可以做清理（释放资源、写日志、更新状态）然后 return，让任务“自己退出”。

对于 **async actor task**（`async def`）：

- `is_canceled()` **不支持**，调用会 `RuntimeError`


detached actor = 集群级常驻 actor：driver 退出也不死；但要你自己负责清理（ray.kill）。

In [10]:
import ray

@ray.remote
class Actor:
    pass

actor_handle = Actor.remote()

ray.kill(actor_handle)
# Force kill: the actor exits immediately without cleanup.
# This will NOT call __ray_shutdown__() or atexit handlers.

In [16]:
import ray
import asyncio

@ray.remote
class AsyncActor:
    def __init__(self, expected_num_tasks: int):
        self._event = asyncio.Event()
        self._curr_num_tasks = 0
        self._expected_num_tasks = expected_num_tasks

    # Multiple invocations of this method can run concurrently on the same event loop.
    async def run_concurrent(self):
        self._curr_num_tasks += 1
        if self._curr_num_tasks == self._expected_num_tasks:
            print("All coroutines are executing concurrently, unblocking.")
            self._event.set()
        else:
            print("Waiting for other coroutines to start.")

        await self._event.wait()
        print("All coroutines ran concurrently.")

actor = AsyncActor.remote(4)
refs = [actor.run_concurrent.remote() for _ in range(4)]

# Fetch results using regular `ray.get`.
# ray.get(refs)

# Fetch results using `asyncio` APIs.
async def get_async():
    return await asyncio.gather(*refs)
# asyncio.run(get_async())

await get_async()

[None, None, None, None]

(AsyncActor pid=14674) Waiting for other coroutines to start.
(AsyncActor pid=14674) Waiting for other coroutines to start.
(AsyncActor pid=14674) Waiting for other coroutines to start.
(AsyncActor pid=14674) All coroutines are executing concurrently, unblocking.
(AsyncActor pid=14674) All coroutines ran concurrently.
(AsyncActor pid=14674) All coroutines ran concurrently.
(AsyncActor pid=14674) All coroutines ran concurrently.
(AsyncActor pid=14674) All coroutines ran concurrently.


In [22]:
import ray
import asyncio

@ray.remote
def some_task():
    return 1

async def convert_to_asyncio_future():
    ref = some_task.remote()
    fut: asyncio.Future = asyncio.wrap_future(ref.future())
    print(await fut)

await convert_to_asyncio_future()

1


In [23]:
import concurrent

refs = [some_task.remote() for _ in range(4)]
futs = [ref.future() for ref in refs]
for fut in concurrent.futures.as_completed(futs):
    assert fut.done()
    print(fut.result())

1
1
1
1


In [24]:
import ray
import asyncio


@ray.remote
class AsyncActor:
    def __init__(self, expected_num_tasks: int):
        self._event = asyncio.Event()
        self._curr_num_tasks = 0
        self._expected_num_tasks = expected_num_tasks

    async def run_task(self):
        print("Started task")
        self._curr_num_tasks += 1
        if self._curr_num_tasks == self._expected_num_tasks:
            self._event.set()
        else:
            # Yield the event loop for multiple coroutines to run concurrently.
            await self._event.wait()

        print("Finished task")

actor = AsyncActor.remote(5)
# All 5 tasks will start at once and run concurrently.
ray.get([actor.run_task.remote() for _ in range(5)])

[None, None, None, None, None]

(AsyncActor pid=18725) Started task
(AsyncActor pid=18725) Started task
(AsyncActor pid=18725) Started task
(AsyncActor pid=18725) Started task
(AsyncActor pid=18725) Started task
(AsyncActor pid=18725) Finished task
(AsyncActor pid=18725) Finished task
(AsyncActor pid=18725) Finished task
(AsyncActor pid=18725) Finished task
(AsyncActor pid=18725) Finished task


In [27]:
@ray.remote
class ThreadedActor:
    def task_1(self): print("I'm running in a thread!")
    def task_2(self): print("I'm running in another thread!")

a = ThreadedActor.options(max_concurrency=2).remote()
ray.get([a.task_1.remote(), a.task_2.remote()])

[None, None]

(ThreadedActor pid=18727) I'm running in another thread!
(ThreadedActor pid=18727) I'm running in a thread!


In [28]:
import ray

@ray.remote(concurrency_groups={"io": 2, "compute": 4})
class AsyncIOActor:
    def __init__(self):
        pass

    @ray.method(concurrency_group="io")
    async def f1(self):
        pass

    @ray.method(concurrency_group="io")
    async def f2(self):
        pass

    @ray.method(concurrency_group="compute")
    async def f3(self):
        pass

    @ray.method(concurrency_group="compute")
    async def f4(self):
        pass

    async def f5(self):
        pass

a = AsyncIOActor.remote()
a.f1.remote()  # executed in the "io" group.
a.f2.remote()  # executed in the "io" group.
a.f3.remote()  # executed in the "compute" group.
a.f4.remote()  # executed in the "compute" group.
a.f5.remote()  # executed in the default group.

ObjectRef(e7e9e65da7da64ad7e8b465ecd197bdbd1e123c70100000001000000)

In [29]:
@ray.remote(concurrency_groups={"io": 2})
class AsyncIOActor:
    async def f1(self):
        pass

actor = AsyncIOActor.options(max_concurrency=10).remote()

In [30]:
import ray
from ray.util import ActorPool


@ray.remote
class Actor:
    def double(self, n):
        return n * 2


a1, a2 = Actor.remote(), Actor.remote()
pool = ActorPool([a1, a2])

# pool.map(..) returns a Python generator object ActorPool.map
gen = pool.map(lambda a, v: a.double.remote(v), [1, 2, 3, 4])
print(list(gen))
# [2, 4, 6, 8]

[2, 4, 6, 8]


In [32]:
import ray
from ray.util.queue import Queue, Empty

# ray.init()
# You can pass this object around to different tasks/actors
queue = Queue(maxsize=100)


@ray.remote
def consumer(id, queue):
    try:
        while True:
            next_item = queue.get(block=True, timeout=1)
            print(f"consumer {id} got work {next_item}")
    except Empty:
        pass


[queue.put(i) for i in range(10)]
print("Put work 1 - 10 to queue...")

consumers = [consumer.remote(id, queue) for id in range(2)]
ray.get(consumers)

Put work 1 - 10 to queue...
(consumer pid=14436) consumer 1 got work 0
(consumer pid=14436) consumer 1 got work 2


[None, None]

In [33]:
import time
import ray

@ray.remote
class Counter:
    def __init__(self):
        self.value = 0

    def add(self, addition):
        self.value += addition
        return self.value

counter = Counter.remote()

# Submit task from a worker
@ray.remote
def submitter(value):
    return ray.get(counter.add.remote(value))

# Simulate delayed result resolution.
@ray.remote
def delayed_resolution(value):
    time.sleep(1)
    return value

# Submit tasks from different workers, with
# the first submitted task waiting for
# dependency resolution.
value0 = submitter.remote(delayed_resolution.remote(1))
value1 = submitter.remote(2)

# Output: 3. The first submitted task is executed later.
print(ray.get(value0))
# Output: 2. The later submitted task is executed first.
print(ray.get(value1))

3
2
